In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader 
from torchvision import datasets, transforms

In [2]:
!nvidia-smi

Mon Jul 27 09:43:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   74C    P0             31W /   70W |     267MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# transform to tensor
transform = transforms.Compose([
   transforms.ToTensor(), # convert to tensor
   transforms.Normalize((0.1307,), (0.3081,)) # normalize the data (Global mean and std for  MNIST)
])

# split data
train_data = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_data  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

# create dataloader
train_loader = DataLoader(dataset=train_data, batch_size=16, num_workers=2, shuffle=True)
test_loader = DataLoader(dataset=test_data, batch_size=1000, num_workers=2, shuffle=False)

In [4]:
# connected colab from vs code (so external files can't be loaded)
import torch.nn as nn

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear( 32 * 7 * 7, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes),
        )
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [5]:
import torch.optim as optim
 
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SimpleCNN().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [6]:
best_ac = 0.0

# evaluation definition
@torch.no_grad()
def evaluate():
    model.eval()
    tp_fn, total  =  0, 0
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        pred = outputs.argmax(dim=1)
        tp_fn += (pred == labels).sum().item()
        total += labels.size(0)
    return tp_fn / total


# training definition
def train(epoch:int=5):
    global best_ac
    model.train()
    for i in range(epoch):
        for image, label in train_loader:
            image, label = image.to(device), label.to(device)
            # clear gradients
            optimizer.zero_grad()
            output = model(image)
            loss = loss_fn(output, label)
            loss.backward()
            optimizer.step()
        acc = evaluate()
        print(f"Epoch-{i + 1} - Test Accuracy: {acc}")
        # save model with best accuracy
        if acc > best_ac:
            torch.save(model.state_dict(), "model.pth")
            best_ac = acc

In [7]:
train(5)

Epoch-1 - Test Accuracy: 0.9834
Epoch-2 - Test Accuracy: 0.9854
Epoch-3 - Test Accuracy: 0.988
Epoch-4 - Test Accuracy: 0.989
Epoch-5 - Test Accuracy: 0.9869


In [8]:
# save the model to gdrive
from google.colab import drive
drive.mount('/content/drive', True)

torch.save(model.state_dict(), "/content/drive/MyDrive/best-model.pth")

Mounted at /content/drive


In [5]:
# make calibration dataset (save to google drive and download)
import os
import numpy as np

calib_dir = "calib"
os.makedirs(calib_dir, exist_ok=True)
num_calib_samples = 50

for i in range(num_calib_samples):
    sample, _ = test_data[i]
    sample_np = sample.numpy().astype(np.float32)
    np.save(os.path.join(calib_dir, f"{i}.npy"), sample_np)
print(f"Saved {num_calib_samples} calibration samples to '{calib_dir}/'")

Saved 50 calibration samples to 'calib/'


In [8]:
# check after conversion
for i in range(num_calib_samples):
    arr = np.load(os.path.join(calib_dir, f"{i}.npy"))
    assert arr.shape == (1, 28, 28), f"Unexpected shape at {i}: {arr.shape}"
    assert arr.dtype == np.float32, f"Unexpected dtype at {i}: {arr.dtype}"
    assert not np.isnan(arr).any(), f"NaN found in sample {i}"
    assert not np.isinf(arr).any(), f"Inf found in sample {i}"

In [7]:
# zip and copy to Drive
import shutil

shutil.make_archive("calib", "zip", ".", calib_dir)
shutil.copy("calib.zip", "/content/drive/MyDrive/calib.zip")
print("calib.zip saved to Google Drive.")

calib.zip saved to Google Drive.
